# Wild and Wonderful Windbarbs Workflow Walkthrough

This runs `windbard-workflow-test.yaml` and shows the intermediate workflow-level **DataTree** state after each plugin finishes.  We are runing the workflow loop by hand so we can inspect the OBP machinery... but you would never do this in real life.

Setup the env

In [1]:
from __future__ import annotations
import os
import yaml
from typing import Any
from datetime import datetime, timezone
import xarray as xr
from IPython.display import display, Markdown, HTML

# Set output directory so GeoIPS does not cry out in pain
os.environ.setdefault("GEOIPS_OUTDIRS", os.path.expanduser("~/GEOIPS_OUTDIRS"))
from geoips.pydantic_models.v1.workflows import WorkflowPluginModel
from geoips.interfaces.class_based.workflow import (
    Workflow,
    INPUT_REF,
    compute_arguments_hash,
    compute_step_output_token,
    StepProvenance,
)
from geoips.utils.types.datatree_ditto import DataTreeDitto

## Load and validate workflow

In [2]:
with open("windbard-workflow-test.yaml") as file:
    the_good_stuff = yaml.safe_load(file)

workflow_model = WorkflowPluginModel(**the_good_stuff, is_registered=False)
spec = workflow_model.spec

display(Markdown(
    f"**Workflow:** `{workflow_model.name}`\n\n"
    f"**Family:** `{workflow_model.family}`\n\n"
    f"**Steps:** {list(spec.steps.keys())}"
))

# expanded step definitions with depends_on defaults.
for sid, step in spec.steps.items():
    deps = list(step.depends_on or [])
    print(f"  {sid:24s}  kind={step.kind:18s}  name={step.name:22s}  deps={deps}")

**Workflow:** `abi.static.dmw.imagery.windbarbs`

**Family:** `order_based`

**Steps:** ['sector', 'reader', 'interpolator', 'algorithm', 'coverage_checker', 'colormapper', 'filename_formatter', 'output_formatter']

  sector                    kind=sector              name=goes_east               deps=[]
  reader                    kind=reader              name=abi_l2_netcdf           deps=['sector']
  interpolator              kind=interpolator        name=interp_gauss            deps=['reader', 'sector']
  algorithm                 kind=algorithm           name=windbarbs_dmw           deps=['interpolator']
  coverage_checker          kind=coverage_checker    name=windbarbs               deps=['algorithm', 'sector']
  colormapper               kind=colormapper         name=matplotlib_linear_norm  deps=['coverage_checker']
  filename_formatter        kind=filename_formatter  name=geoips_fname            deps=['algorithm', 'coverage_checker', 'sector']
  output_formatter          kind=output_formatter    name=imagery_windbarbs       deps=['algorithm', 'colormapper', 'filename_formatter', 'sector']


## A hacky Workflow executor

The `Workflow` class computes the ordering of the DAG for its steps

In [3]:
wf = Workflow(spec, workflow_name=workflow_model.name)

display(Markdown(
    f"**Executed in this order:** `{' → '.join(wf._order)}`\n\n"
    f"**Root steps:** `{wf._entry_steps}`\n\n"
))

**Executed in this order:** `sector → reader → interpolator → algorithm → coverage_checker → colormapper → filename_formatter → output_formatter`

**Root steps:** `{'sector'}`



## Prepare the initial input DataTree

In [4]:
# The data files we will read.... sorry this is just hard-coded. But you can change to whatever!
FILENAMES = [
    "/tmp/test_data_abi/test_data_abi/data/goes16_20210929_0000/"
    "OR_ABI-L2-DMWVF-M6C08_G16_s20212720000206_e20212720009514_c20212720043313.nc",
]

tree = xr.DataTree(name=workflow_model.name)
injected: dict[str, xr.DataTree] = {}  # top-level does not have an injection from the parent

# provenance
wf._set_root_attrs(tree, datetime.now(timezone.utc).isoformat())
executed: set[str] = set()

print(f"Current attrs: {dict(tree.attrs)}")

Current attrs: {'workflow_name': 'abi.static.dmw.imagery.windbarbs', 'outputs': ['output_formatter'], 'retention_policy': 'keep_referenced', 'geoips_version': '0.0.0', 'api_version': 'geoips/v1', 'start_time': '2026-07-07T21:32:20.545288+00:00'}


## A little friend to help us display a DataTree node

In [5]:
def _node_summary(node):
    """Summarise a single workflow-level DataTree node."""
    name = node.name or "<unnamed>"
    ds = node.ds
    if ds is None:
        return f"**{name}**  *(no ds)*"

    pkind = ds.attrs.get("plugin_kind", "")
    pname = ds.attrs.get("plugin_name", "")
    cov   = ds.attrs.get("coverage")
    token = ds.attrs.get("output_token", "")
    dv    = list(ds.data_vars.keys())
    coords = list(ds.coords.keys())
    children = list(node.children.keys())

    badges = []
    if pkind:
        badges.append(f"kind={pkind}")
    if pname:
        badges.append(f"plugin={pname}")
    if cov is not None:
        badges.append(f"coverage={cov}%")
    if token:
        badges.append(f"token={token[:14]}...")

    parts = [f"**{name}**"]
    if badges:
        parts.append(f"({chr(44).join(badges)})")
    if dv:
        parts.append(f"  data_vars={dv}")
    if coords:
        parts.append(f"  coords={coords}")
    if children:
        parts.append(f"  sub-nodes={children}")

    return "  ".join(parts)


def show_tree(tree, title):
    """Print the workflow-level DataTree: root + step outputs only."""
    lines = []
    root_attrs = list(dict(tree.attrs).keys()) if tree.attrs else []
    lines.append(f"`ROOT` **{tree.name}**  attrs={root_attrs}")
    for child_name, child_node in dict(tree.children).items():
        lines.append(f"` - STEP` {_node_summary(child_node)}")
    display(Markdown(f"### {title}\n\n" + "\n\n".join(lines)))


## Helper: run one step (basically copied from workflow call)

In [6]:
def run_step(sid: str) -> tuple[str, Any, xr.DataTree, xr.DataTree]:
    """Execute one workflow step and return (sid, upstream, result, full_tree)."""
    step_def = spec.steps[sid]
    is_entry = sid in wf._entry_steps

    arg_hash = compute_arguments_hash(step_def.arguments or {})
    start_iso = datetime.now(timezone.utc).isoformat()

    upstream = wf._collect_upstream_data(
        tree, step_def.depends_on or [], injected, is_entry
    )

    step_result = wf._invoke_plugin_step(
        step_def, upstream, is_entry=is_entry, filenames=FILENAMES
    )

    end_iso = datetime.now(timezone.utc).isoformat()

    upstream_tokens: dict[str, str] = {}
    from geoips.interfaces.class_based.workflow import _resolve_ref_node
    for dep in step_def.depends_on or ():
        if dep == INPUT_REF:
            continue
        dep_node = _resolve_ref_node(tree, dep)
        if dep_node is not None and dep_node.attrs:
            token = dep_node.attrs.get("output_token")
            if token:
                upstream_tokens[dep] = token

    output_token = compute_step_output_token(
        step_result,
        plugin_name=step_def.name or sid,
        plugin_kind=step_def.kind,
        arguments=step_def.arguments or {},
        upstream_tokens=upstream_tokens or None,
    )
    prov = StepProvenance(
        plugin_name=step_def.name or sid,
        plugin_kind=step_def.kind,
        start_time=start_iso,
        end_time=end_iso,
        arguments_hash=arg_hash,
        output_token=output_token,
    )

    wf._attach_step_node(tree, sid, step_result)
    wf._record_provenance(tree[sid], prov)
    executed.add(sid)
    wf._apply_retention(tree, executed)

    return sid, upstream, step_result, tree

## Time to actually do the things and run the workflow

The empty input

In [7]:
show_tree(tree, "Root (empty)")

### Root (empty)

`ROOT` **abi.static.dmw.imagery.windbarbs**  attrs=['workflow_name', 'outputs', 'retention_policy', 'geoips_version', 'api_version', 'start_time']

### `sector`

In [8]:
sid, upstream, result, tree = run_step("sector")
display(Markdown(
    f"**Upstream deps:** `{list(upstream.children.keys()) or 'none'}`\n\n"
    f"**Result type:** `{type(result).__name__}`\n\n"
    f"**Key attrs:** `area_administer` = `{result.ds.attrs.get('area_administer', 'N/A')}`"
))
show_tree(tree, "After sector")

**Upstream deps:** `none`

**Result type:** `DataTreeDitto`

**Key attrs:** `area_administer` = `N/A`

### After sector

`ROOT` **abi.static.dmw.imagery.windbarbs**  attrs=['workflow_name', 'outputs', 'retention_policy', 'geoips_version', 'api_version', 'start_time']

` - STEP` **sector**  (kind=sector,plugin=goes_east,token=dask:f87364287...)

### `reader`

Legacy class-based plugin (`family = "standard"`).  The OBP calls it
with `filenames=filenames, data=upstream` and `_pre_call` strips the injected
DataTree because old plugins don't deserve to have a good time

In [9]:
sid, upstream, result, tree = run_step("reader")
display(Markdown(
    f"**Upstream deps:** `{list(upstream.children.keys())}`\n\n"
    f"**Result type:** `{type(result).__name__}`\n\n"
    f"**Result children:** `{list(result.children.keys())}`\n\n"
    f"**Result root data_vars:** `{list(result.ds.data_vars.keys())}`  "
    f"(root has attrs only, data is in children)"
))
show_tree(tree, "After reader")

No filenames found for reader: abi_l2_nc


Don't know how to open the following files: {'/tmp/test_data_abi/test_data_abi/data/goes16_20210929_0000/OR_ABI-L2-DMWVF-M6C08_G16_s20212720000206_e20212720009514_c20212720043313.nc'}


**Upstream deps:** `['sector']`

**Result type:** `DataTree`

**Result children:** `['abi_l2_data']`

**Result root data_vars:** `[]`  (root has attrs only, data is in children)

### After reader

`ROOT` **abi.static.dmw.imagery.windbarbs**  attrs=['workflow_name', 'outputs', 'retention_policy', 'geoips_version', 'api_version', 'start_time']

` - STEP` **sector**  (kind=sector,plugin=goes_east,token=dask:f87364287...)

` - STEP` **reader**  (kind=reader,plugin=abi_l2_netcdf,token=dask:7777e07cc...)    sub-nodes=['abi_l2_data']

### `interpolator`

Depends on `[reader, sector]` — needs both raw data and secotr

In [10]:
sid, upstream, result, tree = run_step("interpolator")
display(Markdown(
    f"**Upstream deps:** `{list(upstream.children.keys())}`\n\n"
    f"**Result type:** `{type(result).__name__}`\n\n"
    f"**Result data_vars:** `{list(result.ds.data_vars.keys())}`  "
    f"(interpolated variables on the area definition grid)"
))
show_tree(tree, "After interpolator")

**Upstream deps:** `['reader', 'sector']`

**Result type:** `DataTreeDitto`

**Result data_vars:** `['wind_speed', 'wind_direction', 'pressure', 'latitude', 'longitude']`  (interpolated variables on the area definition grid)

### After interpolator

`ROOT` **abi.static.dmw.imagery.windbarbs**  attrs=['workflow_name', 'outputs', 'retention_policy', 'geoips_version', 'api_version', 'start_time']

` - STEP` **sector**  (kind=sector,plugin=goes_east,token=dask:f87364287...)

` - STEP` **reader**  (kind=reader,plugin=abi_l2_netcdf,token=dask:7777e07cc...)    sub-nodes=['abi_l2_data']

` - STEP` **interpolator**  (kind=interpolator,plugin=interp_gauss,token=dask:9892de4cb...)    data_vars=['wind_speed', 'wind_direction', 'pressure', 'latitude', 'longitude']

### wimdy `algorithm`

Depends on just `interpolato`

In [11]:
sid, upstream, result, tree = run_step("algorithm")
display(Markdown(
    f"**Upstream deps:** `{list(upstream.children.keys())}`\n\n"
    f"**Result type:** `{type(result).__name__}`\n\n"
    f"**Result data_vars:** `{list(result.ds.data_vars.keys())}`\n\n"
    f"**product_name in attrs:** `{result.ds.attrs.get('product_name')}`"
))
show_tree(tree, "After algorithm")

**Upstream deps:** `['interpolator']`

**Result type:** `DataTreeDitto`

**Result data_vars:** `['wind_speed', 'wind_direction', 'pressure', 'latitude', 'longitude']`

**product_name in attrs:** `DMW-High`

### After algorithm

`ROOT` **abi.static.dmw.imagery.windbarbs**  attrs=['workflow_name', 'outputs', 'retention_policy', 'geoips_version', 'api_version', 'start_time']

` - STEP` **sector**  (kind=sector,plugin=goes_east,token=dask:f87364287...)

` - STEP` **reader**  (kind=reader,plugin=abi_l2_netcdf,token=dask:7777e07cc...)    sub-nodes=['abi_l2_data']

` - STEP` **interpolator**  (kind=interpolator,plugin=interp_gauss,token=dask:9892de4cb...)    data_vars=['wind_speed', 'wind_direction', 'pressure', 'latitude', 'longitude']

` - STEP` **algorithm**  (kind=algorithm,plugin=windbarbs_dmw,token=dask:c6e934338...)    data_vars=['wind_speed', 'wind_direction', 'pressure', 'latitude', 'longitude']    coords=['DMW-High']

### `coverage_checker`

Depends on `[algorithm, sector]`.  Receives a `variables` list in
step arguments → the `_invoke` override in `BaseCoverageCheckerPlugin`
**iterates** over each variable, and calls the plugin once per name with
`variable_name=<var>` injected.  If `minimum_coverage` is set and any
variable's coverage falls below it, a `ValueError` is raised
immediately

In [12]:
sid, upstream, result, tree = run_step("coverage_checker")
display(Markdown(
    f"**Upstream deps:** `{list(upstream.children.keys())}`\n\n"
    f"**Result type:** `{type(result).__name__}`\n\n"
    f"**Coverage:** `{result.ds.attrs.get('coverage', 'N/A')}%`  "
    f"(minimum across all checked variables)"
))
show_tree(tree, "After coverage_checker")

**Upstream deps:** `['algorithm', 'sector']`

**Result type:** `DataTreeDitto`

**Coverage:** `0.0%`  (minimum across all checked variables)

### After coverage_checker

`ROOT` **abi.static.dmw.imagery.windbarbs**  attrs=['workflow_name', 'outputs', 'retention_policy', 'geoips_version', 'api_version', 'start_time']

` - STEP` **sector**  (kind=sector,plugin=goes_east,token=dask:f87364287...)

` - STEP` **reader**  (kind=reader,plugin=abi_l2_netcdf,token=dask:7777e07cc...)    sub-nodes=['abi_l2_data']

` - STEP` **interpolator**  (kind=interpolator,plugin=interp_gauss,token=dask:9892de4cb...)    data_vars=['wind_speed', 'wind_direction', 'pressure', 'latitude', 'longitude']

` - STEP` **algorithm**  (kind=algorithm,plugin=windbarbs_dmw,token=dask:c6e934338...)    data_vars=['wind_speed', 'wind_direction', 'pressure', 'latitude', 'longitude']    coords=['DMW-High']

` - STEP` **coverage_checker**  (kind=coverage_checker,plugin=windbarbs,coverage=0.0%,token=dask:c3f1e6041...)

### `colormapper` 
`_post_call` wraps the dict into `DataTreeDitto`

In [13]:
sid, upstream, result, tree = run_step("colormapper")
display(Markdown(
    f"**Upstream deps:** `{list(upstream.children.keys())}`\n\n"
    f"**Result type:** `{type(result).__name__}`\n\n"
    f"**_mpl_colors_info keys:** "
    f"`{list(result.ds.attrs.get('_mpl_colors_info', {}).keys())}`"
))
show_tree(tree, "After colormapper")

**Upstream deps:** `['coverage_checker']`

**Result type:** `DataTreeDitto`

**_mpl_colors_info keys:** `['cmap', 'norm', 'boundaries', 'colorbar', 'cbar_ticks', 'cbar_tick_labels', 'cbar_label', 'cbar_spacing', 'cbar_full_width', 'colorbar_kwargs', 'set_ticks_kwargs', 'set_label_kwargs']`

### After colormapper

`ROOT` **abi.static.dmw.imagery.windbarbs**  attrs=['workflow_name', 'outputs', 'retention_policy', 'geoips_version', 'api_version', 'start_time']

` - STEP` **sector**  (kind=sector,plugin=goes_east,token=dask:f87364287...)

` - STEP` **reader**  (kind=reader,plugin=abi_l2_netcdf,token=dask:7777e07cc...)    sub-nodes=['abi_l2_data']

` - STEP` **interpolator**  (kind=interpolator,plugin=interp_gauss,token=dask:9892de4cb...)    data_vars=['wind_speed', 'wind_direction', 'pressure', 'latitude', 'longitude']

` - STEP` **algorithm**  (kind=algorithm,plugin=windbarbs_dmw,token=dask:c6e934338...)    data_vars=['wind_speed', 'wind_direction', 'pressure', 'latitude', 'longitude']    coords=['DMW-High']

` - STEP` **coverage_checker**  (kind=coverage_checker,plugin=windbarbs,coverage=0.0%,token=dask:c3f1e6041...)

` - STEP` **colormapper**  (kind=colormapper,plugin=matplotlib_linear_norm,token=dask:43669db33...)

###`filename_formatter`

Depends on `[algorithm, coverage_checker, sector]` and its `_post_call` wraps the path string into `DataTreeDitto` attrs

In [14]:
sid, upstream, result, tree = run_step("filename_formatter")
display(Markdown(
    f"**Upstream deps:** `{list(upstream.children.keys())}`\n\n"
    f"**Result type:** `{type(result).__name__}`\n\n"
    f"**Output filenames:** `{result.ds.attrs.get('output_filenames')}`"
))
show_tree(tree, "After filename_formatter")

**Upstream deps:** `['algorithm', 'coverage_checker', 'sector']`

**Result type:** `DataTreeDitto`

**Output filenames:** `['/Users/gwynu/GEOIPS_OUTDIRS/preprocessed/annotated_imagery/Global-x-GOES-East/x-x-x/DMW-High/abi/20210929.000020.goes-16.abi.DMW-High.goes_east.0p00.noaa.10p0.png']`

### After filename_formatter

`ROOT` **abi.static.dmw.imagery.windbarbs**  attrs=['workflow_name', 'outputs', 'retention_policy', 'geoips_version', 'api_version', 'start_time']

` - STEP` **sector**  (kind=sector,plugin=goes_east,token=dask:f87364287...)

` - STEP` **reader**  (kind=reader,plugin=abi_l2_netcdf,token=dask:7777e07cc...)    sub-nodes=['abi_l2_data']

` - STEP` **interpolator**  (kind=interpolator,plugin=interp_gauss,token=dask:9892de4cb...)    data_vars=['wind_speed', 'wind_direction', 'pressure', 'latitude', 'longitude']

` - STEP` **algorithm**  (kind=algorithm,plugin=windbarbs_dmw,token=dask:c6e934338...)    data_vars=['wind_speed', 'wind_direction', 'pressure', 'latitude', 'longitude']    coords=['DMW-High']

` - STEP` **coverage_checker**  (kind=coverage_checker,plugin=windbarbs,coverage=0.0%,token=dask:c3f1e6041...)

` - STEP` **colormapper**  (kind=colormapper,plugin=matplotlib_linear_norm,token=dask:43669db33...)

` - STEP` **filename_formatter**  (kind=filename_formatter,plugin=geoips_fname,token=dask:7a1d9525f...)    data_vars=['output_path']

### `output_formatter`

Final step (yayyy!!!!) Depends on `[algorithm, colormapper, filename_formatter,
sector]` so a total of 4 upstream branches!! So many! Takes a arg for the product name so if you want to override whats set in the algorithm you can

In [15]:
sid, upstream, result, tree = run_step("output_formatter")
display(Markdown(
    f"**Upstream deps:** `{list(upstream.children.keys())}`\n\n"
    f"**Result type:** `{type(result).__name__}`\n\n"
    f"**Output file:** `{result.ds.attrs.get('output_filenames')}`"
))
show_tree(tree, "After output_formatter (final tree)")

**Upstream deps:** `['algorithm', 'colormapper', 'filename_formatter', 'sector']`

**Result type:** `DataTreeDitto`

**Output file:** `None`

### After output_formatter (final tree)

`ROOT` **abi.static.dmw.imagery.windbarbs**  attrs=['workflow_name', 'outputs', 'retention_policy', 'geoips_version', 'api_version', 'start_time']

` - STEP` **sector**  (kind=sector,plugin=goes_east,token=dask:f87364287...)

` - STEP` **reader**  (kind=reader,plugin=abi_l2_netcdf,token=dask:7777e07cc...)    sub-nodes=['abi_l2_data']

` - STEP` **interpolator**  (kind=interpolator,plugin=interp_gauss,token=dask:9892de4cb...)    data_vars=['wind_speed', 'wind_direction', 'pressure', 'latitude', 'longitude']

` - STEP` **algorithm**  (kind=algorithm,plugin=windbarbs_dmw,token=dask:c6e934338...)    data_vars=['wind_speed', 'wind_direction', 'pressure', 'latitude', 'longitude']    coords=['DMW-High']

` - STEP` **coverage_checker**  (kind=coverage_checker,plugin=windbarbs,coverage=0.0%,token=dask:c3f1e6041...)

` - STEP` **colormapper**  (kind=colormapper,plugin=matplotlib_linear_norm,token=dask:43669db33...)

` - STEP` **filename_formatter**  (kind=filename_formatter,plugin=geoips_fname,token=dask:7a1d9525f...)    data_vars=['output_path']

` - STEP` **output_formatter**  (kind=output_formatter,plugin=imagery_windbarbs,token=dask:8292daba1...)